# Notebook 03: Evaluasi Baseline RAG — PubMedQA

Konfigurasi **Baseline RAG** tanpa teknik mitigasi apapun.
Retriever: **BM25 (keyword-based)**, tidak memerlukan embedding saat indexing.

## Konfigurasi
| Parameter | Nilai |
|-----------|-------|
| Query Rewriting | Tidak |
| Context Reranking | Tidak |
| Active Detection | Tidak |
| Retriever | BM25 (rank\_bm25) |
| LLM | llama3.2 via Ollama |
| Embedding | nomic-embed-text (hanya untuk RAGAS Answer Relevancy) |
| Dataset | PubMedQA pqa\_labeled |
| Sampel | 500 — ubah ke 1000 untuk evaluasi final |

## Alur
1. **DEMO** 5 sampel (verifikasi pipeline)
2. **Fase 1** Generate 500 jawaban + label accuracy (cepat)
3. **Fase 2** Evaluasi RAGAS per sampel (lambat, bisa semalam)
4. **Analisis** Ringkasan metrik + baris tabel skripsi

## 1. Impor Library

In [11]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime
from collections import Counter

from rank_bm25 import BM25Okapi
import ollama
from datasets import load_dataset

# RAGAS dengan Ollama — gunakan old-style metrics (_Faithfulness, dll.)
# karena ragas.metrics.collections baru hanya support InstructorLLM (OpenAI)
warnings.filterwarnings('ignore', category=DeprecationWarning)
from ragas import EvaluationDataset, SingleTurnSample, evaluate, RunConfig
from ragas.metrics import (
    _Faithfulness,
    _ResponseRelevancy,
    _LLMContextPrecisionWithReference,
    _LLMContextRecall
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaLLM, OllamaEmbeddings

print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | Pandas: {pd.__version__}')

Semua library berhasil diimpor!
Python: 3.11.9 | NumPy: 2.3.5 | Pandas: 2.3.3


## 2. Konfigurasi

In [12]:
LLM_MODEL   = 'llama3.2'
EMBED_MODEL = 'nomic-embed-text'  # Hanya untuk RAGAS, bukan retrieval

TOP_K_RETRIEVAL = 5

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'

# CATATAN: Mulai 500. UBAH KE 1000 untuk evaluasi final thesis!
MAX_SAMPLES = 500

TEMPERATURE = 0.0
SEED        = 42

NOTEBOOK_DIR    = Path('.')
BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'
RESULTS_DIR     = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

CONFIG_NAME    = 'baseline'
PHASE1_PATH    = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'
PHASE2_PATH    = RESULTS_DIR / f'{CONFIG_NAME}_phase2_ragas.json'
FINAL_CSV_PATH = RESULTS_DIR / f'{CONFIG_NAME}_results.csv'

print('Konfigurasi:')
print(f'  LLM        : {LLM_MODEL}')
print(f'  Embedding  : {EMBED_MODEL} (RAGAS only)')
print(f'  Retriever  : BM25')
print(f'  Top-K      : {TOP_K_RETRIEVAL}')
print(f'  Sampel     : {MAX_SAMPLES}')
print()
print('=' * 50)
print(f'  PENGINGAT: MAX_SAMPLES = {MAX_SAMPLES}')
print('  Ubah ke 1000 untuk evaluasi final thesis!')
print('=' * 50)

Konfigurasi:
  LLM        : llama3.2
  Embedding  : nomic-embed-text (RAGAS only)
  Retriever  : BM25
  Top-K      : 5
  Sampel     : 500

  PENGINGAT: MAX_SAMPLES = 500
  Ubah ke 1000 untuk evaluasi final thesis!


## 3. Data Classes dan Tokenizer BM25

In [13]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str

@dataclass
class RetrievalResult:
    document: Document
    score   : float


def tokenize_bm25(text: str) -> List[str]:
    """Tokenizer untuk BM25: hapus tanda baca, lowercase, split spasi."""
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()


# Test
sample_text = 'Does aspirin (75mg) reduce myocardial infarction risk?'
print(f'Tokenisasi BM25: {tokenize_bm25(sample_text)}')
print('Data classes dan tokenizer siap.')

Tokenisasi BM25: ['does', 'aspirin', '75mg', 'reduce', 'myocardial', 'infarction', 'risk']
Data classes dan tokenizer siap.


## 4. Muat Dataset dan Bangun BM25 Index

BM25 tidak memerlukan embedding untuk indexing — jauh lebih cepat dari FAISS.
Index dibangun dari tokenisasi teks dokumen saja.

In [14]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    """Ubah dataset PubMedQA menjadi list Document (satu per section abstrak)."""
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    print(f'Total potongan dokumen: {len(docs)}')
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    """Muat BM25 index dari file jika ada, atau bangun dari scratch."""
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        print(f'Index disimpan ke {BM25_INDEX_PATH}')
        return bm25, documents


t0 = time.time()
pubmedqa_data         = load_pubmedqa()
bm25_index, documents = load_or_build_bm25(pubmedqa_data)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memuat PubMedQA (pqa_labeled)...
Dimuat 500 sampel
Membangun BM25 index...
Total potongan dokumen: 1706
Index disimpan ke pubmedqa_bm25.pkl
Selesai dalam 5.2 detik


## 5. Fungsi Retrieval Baseline (BM25, Tanpa Query Rewriting)

In [15]:
def retrieve_baseline(query: str, k: int = TOP_K_RETRIEVAL) -> List[RetrievalResult]:
    """
    Retrieval baseline menggunakan BM25 (exact keyword matching).
    Tidak ada query rewriting. Tidak ada reranking.

    Catatan untuk notebook Context Reranking nanti:
      retrieve_baseline(query, k=20) untuk ambil lebih banyak kandidat,
      kemudian CrossEncoder rerank ke top-5.
    """
    tokens = tokenize_bm25(query)
    scores = bm25_index.get_scores(tokens)
    top_k  = np.argsort(scores)[::-1][:k]
    return [RetrievalResult(document=documents[i], score=float(scores[i])) for i in top_k]


# Test retrieval
test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r = retrieve_baseline(test_q)
print(f'Query: {test_q}')
print(f'Top-{TOP_K_RETRIEVAL} dokumen (BM25):')
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] Score={r.score:.4f} | {r.document.section_label} | {r.document.text[:90]}...')

Query: Does aspirin reduce the risk of myocardial infarction?
Top-5 dokumen (BM25):
  [1] Score=23.4945 | DESIGN | Within a prospective, population-based cohort study individuals without history of myocard...
  [2] Score=20.3167 | METHODS | By use of the Cooperative Cardiovascular Project database (a retrospective medical record ...
  [3] Score=19.3056 | METHODS | Of the 9681 women and 8888 men who attended risk assessment from 1967-1991, with follow-up...
  [4] Score=18.0483 | STUDY DESIGN | All patients between the ages of 80 to 89 years undergoing carotid endarterectomy during a...
  [5] Score=17.2286 | OBJECTIVE | To examine the effect of a weekend hospitalization on the timing and incidence of intensiv...


## 6. Prompt Generasi dan Fungsi Generate

In [16]:
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Assess whether the context supports, refutes, or is inconclusive about the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence clearly supports the hypothesis\n'
    '  - no    : the evidence clearly refutes the hypothesis\n'
    '  - maybe : the evidence is mixed, insufficient, or inconclusive\n\n'
    'Answer:'
)


def generate_baseline_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    """Generate jawaban. Baseline: tanpa query rewriting, tanpa reranking."""
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    response = ollama.generate(
        model=LLM_MODEL,
        prompt=GENERATION_PROMPT.format(context=context, question=query),
        options={'temperature': TEMPERATURE, 'seed': SEED, 'num_predict': 300}
    )
    return response['response'].strip()


# Test
test_ans = generate_baseline_answer(test_q, test_r)
print('Output generation:')
print('-' * 60)
print(test_ans)
print('-' * 60)

Output generation:
------------------------------------------------------------
The context does not provide direct information about the effect of aspirin on myocardial infarction risk. However, it mentions that individuals without a history of myocardial infarction were assessed for verified myocardial infarction or coronary death during follow-up in study [1], and that 706 women and 1700 men suffered a non-fatal myocardial infarction or coronary death between 1967-1991 in study [3]. There is no mention of aspirin use or its impact on myocardial infarction risk.

maybe
------------------------------------------------------------


## 7. Ekstraksi Label yes/no/maybe

In [17]:
def extract_label(answer: str) -> str:
    """
    Ekstrak prediksi yes/no/maybe dari teks jawaban.
    Strategi (berurutan hingga ditemukan):
      1. Kata standalone di 3 baris terakhir (non-kosong)
      2. Kata standalone di seluruh teks
      3. Default ke 'maybe'
    """
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'


cases = [
    ('Strong evidence.\nyes',    'yes'),
    ('No effect found.\nno',     'no'),
    ('Mixed results.\nmaybe',    'maybe'),
    ('Verdict: yes.',             'yes'),
    ('Totally unclear.',          'maybe'),
]
print('Unit test extract_label:')
all_ok = True
for txt, exp in cases:
    pred = extract_label(txt)
    ok   = pred == exp
    all_ok = all_ok and ok
    print(f'  [{"PASS" if ok else "FAIL"}] pred={pred!r} expected={exp!r}')
print(f'\nSemua lulus: {all_ok}')
print(f'Label dari test answer: {extract_label(test_ans)!r}')

Unit test extract_label:
  [PASS] pred='yes' expected='yes'
  [PASS] pred='no' expected='no'
  [PASS] pred='maybe' expected='maybe'
  [PASS] pred='yes' expected='yes'
  [PASS] pred='maybe' expected='maybe'

Semua lulus: True
Label dari test answer: 'maybe'


## 8. Setup RAGAS dengan Ollama (Tanpa OpenAI)

**Perbaikan `TimeoutError`**: RAGAS default `max_workers=16` — artinya 16 request dikirim ke Ollama bersamaan. Ollama hanya handle 1 sekaligus, sehingga request pertama antri dan timeout.

Solusi: `max_workers=1` (eksekusi sekuensial, tidak ada antrean).

In [18]:
ragas_llm = LangchainLLMWrapper(OllamaLLM(model=LLM_MODEL, temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=EMBED_MODEL))

# KUNCI: max_workers=1 mencegah TimeoutError pada Ollama lokal
ragas_run_config = RunConfig(
    timeout     = 300,  # 5 menit per LLM call — Ollama di CPU bisa lambat
    max_retries = 2,
    max_wait    = 60,
    max_workers = 1,    # <-- FIX TimeoutError: eksekusi sekuensial
    log_tenacity= False,
)

faithfulness_m   = _Faithfulness(llm=ragas_llm)
ans_relevancy_m  = _ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb)
ctx_precision_m  = _LLMContextPrecisionWithReference(llm=ragas_llm)
ctx_recall_m     = _LLMContextRecall(llm=ragas_llm)

ALL_METRICS = [faithfulness_m, ans_relevancy_m, ctx_precision_m, ctx_recall_m]

print('RAGAS siap (tanpa OpenAI, max_workers=1):')
for m in ALL_METRICS:
    print(f'  - {m.name}')


def evaluate_ragas_single(question: str, answer: str,
                          contexts: List[str], reference: str) -> Dict:
    """Evaluasi RAGAS untuk 1 sampel. Mengembalikan NaN jika gagal."""
    nan = float('nan')
    sample  = SingleTurnSample(
        user_input=question, response=answer,
        retrieved_contexts=contexts, reference=reference
    )
    dataset = EvaluationDataset(samples=[sample])
    try:
        result = evaluate(
            dataset, metrics=ALL_METRICS,
            llm=ragas_llm, embeddings=ragas_emb,
            run_config=ragas_run_config,
            raise_exceptions=False, show_progress=False
        )
        row = result.to_pandas().iloc[0]
        return {
            'faithfulness'      : float(row.get('faithfulness',                         nan)),
            'answer_relevancy'  : float(row.get('answer_relevancy',                     nan)),
            'context_precision' : float(row.get('llm_context_precision_with_reference', nan)),
            'context_recall'    : float(row.get('context_recall',                       nan)),
        }
    except Exception as e:
        print(f'  [RAGAS Error] {type(e).__name__}: {e}')
        return {'faithfulness': nan, 'answer_relevancy': nan,
                'context_precision': nan, 'context_recall': nan}

RAGAS siap (tanpa OpenAI, max_workers=1):
  - faithfulness
  - answer_relevancy
  - llm_context_precision_with_reference
  - context_recall


---
## DEMO: Uji Coba 5 Sampel

Jalankan ini dulu untuk memverifikasi seluruh pipeline berjalan dengan benar.

> Estimasi: ~3-10 menit (tergantung kecepatan Ollama)

In [19]:
DEMO_SIZE    = 5
demo_results = []

print(f'DEMO: {DEMO_SIZE} sampel pertama')
print('=' * 65)

for i in range(DEMO_SIZE):
    s         = pubmedqa_data[i]
    q         = s['question']
    gt        = s['final_decision']
    ref       = s['long_answer']

    print(f'\n[{i+1}/{DEMO_SIZE}] PubID: {s["pubid"]}')
    print(f'  Q  : {q[:80]}...')
    print(f'  GT : {gt}')

    t0        = time.time()
    retrieved = retrieve_baseline(q)
    t_ret     = time.time() - t0

    t0     = time.time()
    answer = generate_baseline_answer(q, retrieved)
    t_gen  = time.time() - t0

    predicted = extract_label(answer)
    correct   = predicted == gt

    t0      = time.time()
    ctxs    = [r.document.text for r in retrieved]
    scores  = evaluate_ragas_single(q, answer, ctxs, ref)
    t_ragas = time.time() - t0

    demo_results.append({
        'idx': i, 'pubid': str(s['pubid']), 'question': q,
        'ground_truth': gt, 'predicted_label': predicted,
        'is_correct': correct, 'answer': answer,
        'contexts': ctxs, 'reference': ref, **scores,
        't_retrieval': round(t_ret,2), 't_generation': round(t_gen,2), 't_ragas': round(t_ragas,2)
    })

    verdict = 'BENAR' if correct else 'SALAH'
    print(f'  Pred : {predicted} [{verdict}]')
    print(f'  Ans  : {answer[:100]}...')
    print(f'  RAGAS: faith={scores["faithfulness"]:.3f} | '
          f'rel={scores["answer_relevancy"]:.3f} | '
          f'cp={scores["context_precision"]:.3f} | '
          f'cr={scores["context_recall"]:.3f}')
    print(f'  Waktu: ret={t_ret:.1f}s | gen={t_gen:.1f}s | ragas={t_ragas:.1f}s')

n_ok = sum(r['is_correct'] for r in demo_results)
print(f'\n{"="*65}')
print(f'RINGKASAN DEMO ({DEMO_SIZE} sampel):')
print(f'  Label Accuracy    : {n_ok}/{DEMO_SIZE} = {n_ok/DEMO_SIZE:.1%}')
print(f'  Hallucination Rate: {(DEMO_SIZE-n_ok)/DEMO_SIZE:.1%}')
for col in ['faithfulness','answer_relevancy','context_precision','context_recall']:
    print(f'  Avg {col:<22}: {np.nanmean([r[col] for r in demo_results]):.3f}')
print(f'{"="*65}')

DEMO: 5 sampel pertama

[1/5] PubID: 21645374
  Q  : Do mitochondria play a role in remodelling lace plant leaves during programmed c...
  GT : yes


Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000026C59D362C0> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000026C59D362C0> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._

  Pred : maybe [SALAH]
  Ans  : The context suggests that mitochondria may play a role in programmed cell death (PCD) in lace plant ...
  RAGAS: faith=nan | rel=nan | cp=0.950 | cr=0.750
  Waktu: ret=0.0s | gen=59.8s | ragas=846.0s

[2/5] PubID: 16418930
  Q  : Landolt C and snellen e acuity: differences in strabismus amblyopia?...
  GT : no


Exception raised in Job[1]: TimeoutError()


  Pred : maybe [SALAH]
  Ans  : The context suggests that there are small differences between Landolt C acuity (LR) and Snellen E ac...
  RAGAS: faith=0.750 | rel=nan | cp=1.000 | cr=0.800
  Waktu: ret=0.0s | gen=51.2s | ragas=810.6s

[3/5] PubID: 9488747
  Q  : Syncope during bathing in infants, a pediatric form of water-induced urticaria?...
  GT : yes


...

  Pred : maybe [SALAH]
  Ans  : The context suggests that syncope during bathing in infants may be related to water-induced urticari...
  RAGAS: faith=0.800 | rel=nan | cp=nan | cr=0.286
  Waktu: ret=0.0s | gen=55.0s | ragas=805.0s

[4/5] PubID: 17208539
  Q  : Are the long-term results of the transanal pull-through equal to those of the tr...
  GT : no


Exception raised in Job[0]: TimeoutError()


  Pred : maybe [SALAH]
  Ans  : The context does not provide sufficient information to determine whether the long-term results of th...
  RAGAS: faith=nan | rel=0.000 | cp=0.887 | cr=0.750
  Waktu: ret=0.0s | gen=68.3s | ragas=687.1s

[5/5] PubID: 10808977
  Q  : Can tailored interventions increase mammography use among HMO women?...
  GT : yes


Exception raised in Job[0]: OutputParserException(Invalid json output: Given a question and an answer, analyze the complexity of each sentence in the answer. Break down each sentence into one or more fully understandable statements. Ensure that no pronouns are used in any statement. Format the outputs in JSON.
Please return the output in a JSON format that complies with the following schema as specified in JSON Schema:
{"properties": {"statements": {"description": "The generated statements", "items": {"type": "string"}, "title": "Statements", "type": "array"}}, "required": ["statements"], "title": "StatementGeneratorOutput", "type": "object"}Do not use single quotes in your response but double quotes,properly escaped with a backslash.

--------EXAMPLES-----------
Example 1
Input: {
    "question": "Who was Albert Einstein and what is he best known for?",
    "answer": "He was a German-born theoretical physicist, widely acknowledged to be one of the greatest and most influential physici

  Pred : yes [BENAR]
  Ans  : The context suggests that tailored interventions can increase mammography use among HMO women. The s...
  RAGAS: faith=nan | rel=nan | cp=nan | cr=0.000
  Waktu: ret=0.0s | gen=40.8s | ragas=808.4s

RINGKASAN DEMO (5 sampel):
  Label Accuracy    : 1/5 = 20.0%
  Hallucination Rate: 80.0%
  Avg faithfulness          : 0.775
  Avg answer_relevancy      : 0.000
  Avg context_precision     : 0.946
  Avg context_recall        : 0.517


---
## Fase 1: Generate Semua Jawaban (500 Sampel)

Hanya retrieval + generation, tanpa RAGAS. Disimpan inkremental setiap 10 sampel.

> Estimasi: ~25-50 menit untuk 500 sampel

In [ ]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Memulai Fase 1: {MAX_SAMPLES} sampel.')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()
    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']
        retrieved  = retrieve_baseline(q)
        answer     = generate_baseline_answer(q, retrieved)
        predicted  = extract_label(answer)
        phase1_results.append({
            'idx': i, 'pubid': str(s['pubid']), 'question': q,
            'ground_truth': gt, 'predicted_label': predicted,
            'is_correct': predicted == gt, 'answer': answer,
            'contexts': [r.document.text for r in retrieved],
            'reference': ref, 'retrieval_scores': [r.score for r in retrieved],
        })
        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({'config': CONFIG_NAME,
                           'timestamp': datetime.now().isoformat(),
                           'max_samples': MAX_SAMPLES, 'completed': i+1,
                           'results': phase1_results}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Akurasi: {acc:.1%} | pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')
    print(f'\nFase 1 selesai! Disimpan ke {PHASE1_PATH}')
else:
    print(f'Fase 1 sudah selesai ({MAX_SAMPLES} sampel).')

Resume Fase 1: 10/500 sudah selesai.
Memproses 490 sampel tersisa...

  [ 20/500] Akurasi: 20.0% | pred=maybe, gt=yes | ETA 232.1 mnt
  [ 30/500] Akurasi: 20.0% | pred=maybe, gt=yes | ETA 294.0 mnt
  [ 40/500] Akurasi: 22.5% | pred=maybe, gt=no | ETA 328.4 mnt
  [ 50/500] Akurasi: 22.0% | pred=maybe, gt=no | ETA 345.7 mnt
  [ 60/500] Akurasi: 25.0% | pred=maybe, gt=yes | ETA 348.2 mnt
  [ 70/500] Akurasi: 22.9% | pred=maybe, gt=yes | ETA 351.0 mnt
  [ 80/500] Akurasi: 23.8% | pred=maybe, gt=yes | ETA 352.7 mnt
  [ 90/500] Akurasi: 26.7% | pred=maybe, gt=maybe | ETA 349.2 mnt


### Analisis Fase 1 (Label Accuracy)

In [ ]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)
gts       = [r['ground_truth']    for r in results_p1]
preds     = [r['predicted_label'] for r in results_p1]

print(f'ANALISIS FASE 1 — {n} sampel')
print('=' * 50)
print(f'Label Accuracy    : {n_correct}/{n} = {n_correct/n:.1%}')
print(f'Hallucination Rate: {(n-n_correct)/n:.1%}\n')
print(f'  {"Label":<8} | {"Ground Truth":>12} | {"Prediksi":>12}')
print(f'  {"-"*8}-+{"-"*14}-+{"-"*12}')
for lbl in ['yes','no','maybe']:
    g, p = gts.count(lbl), preds.count(lbl)
    print(f'  {lbl:<8} | {g:>10} ({g/n:.0%}) | {p:>10} ({p/n:.0%})')
print('\nConfusion Matrix (baris=GT, kolom=Prediksi):')
lbls = ['yes','no','maybe']
print('  ' + f'{"GT/Pred":>8}' + ''.join(f'{l:>8}' for l in lbls))
for gt_l in lbls:
    row = f'  {gt_l:>8}'
    for pr_l in lbls:
        cnt = sum(1 for r in results_p1 if r['ground_truth']==gt_l and r['predicted_label']==pr_l)
        row += f'{cnt:>8}'
    print(row)

---
## Fase 2: Evaluasi RAGAS

Jalankan setelah Fase 1 selesai. Mendukung resume — aman dihentikan kapan saja.

> Estimasi: ~2-5 menit per sampel. Disarankan dijalankan semalam.

In [ ]:
if PHASE2_PATH.exists():
    with open(PHASE2_PATH, 'r', encoding='utf-8') as f:
        phase2_results = json.load(f)['results']
    done_idx = {r['idx'] for r in phase2_results}
    print(f'Resume Fase 2: {len(done_idx)}/{len(results_p1)} sudah dievaluasi.')
else:
    phase2_results, done_idx = [], set()
    print(f'Memulai Fase 2: {len(results_p1)} sampel.')

remaining = [r for r in results_p1 if r['idx'] not in done_idx]
print(f'Sisa: {len(remaining)} sampel | Estimasi ~{len(remaining)*3/60:.1f} jam\n')

t_p2 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_ragas_single(r['question'], r['answer'], r['contexts'], r['reference'])
    phase2_results.append({
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        **scores
    })
    if (i + 1) % 5 == 0 or i == len(remaining) - 1:
        with open(PHASE2_PATH, 'w', encoding='utf-8') as f:
            json.dump({'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                       'results': phase2_results}, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining)
        eta   = (time.time()-t_p2)/done*(total-done)/60 if done < total else 0
        avg_f = np.nanmean([x['faithfulness'] for x in phase2_results])
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'faith={scores["faithfulness"]:.3f} | avg_f={avg_f:.3f} | ETA {eta:.1f} mnt')
print(f'\nFase 2 selesai! Disimpan ke {PHASE2_PATH}')

---
## Hasil Akhir: Gabungkan Fase 1 + Fase 2 → CSV

In [ ]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
with open(PHASE2_PATH, 'r', encoding='utf-8') as f:
    results_p2 = json.load(f)['results']

p2_lookup = {r['idx']: r for r in results_p2}

df = pd.DataFrame([{
    'idx': r['idx'], 'pubid': r['pubid'], 'question': r['question'],
    'ground_truth': r['ground_truth'], 'predicted_label': r['predicted_label'],
    'is_correct': r['is_correct'],
    'faithfulness'      : p2_lookup.get(r['idx'],{}).get('faithfulness',       float('nan')),
    'answer_relevancy'  : p2_lookup.get(r['idx'],{}).get('answer_relevancy',   float('nan')),
    'context_precision' : p2_lookup.get(r['idx'],{}).get('context_precision',  float('nan')),
    'context_recall'    : p2_lookup.get(r['idx'],{}).get('context_recall',     float('nan')),
} for r in results_p1])

df.to_csv(FINAL_CSV_PATH, index=False, encoding='utf-8')
print(f'Disimpan ke: {FINAL_CSV_PATH}')
print(f'Total: {len(df)} | RAGAS lengkap: {df["faithfulness"].notna().sum()}')
df.head(10)

## Ringkasan Metrik Evaluasi Baseline

In [ ]:
n, n_correct = len(df), int(df['is_correct'].sum())
label_acc  = n_correct / n
hallu_rate = 1 - label_acc
avg_f  = df['faithfulness'].mean()
avg_r  = df['answer_relevancy'].mean()
avg_cp = df['context_precision'].mean()
avg_cr = df['context_recall'].mean()

print('=' * 60)
print(f'  BASELINE (BM25) — {n} sampel')
print('=' * 60)
print(f'  {"Metrik":<32} {"Nilai":>10}')
print(f'  {"-"*42}')
print(f'  {"Label Accuracy":<32} {label_acc:>9.1%}')
print(f'  {"Hallucination Rate":<32} {hallu_rate:>9.1%}')
print(f'  {"-"*42}')
print(f'  {"Faithfulness (RAGAS)":<32} {avg_f:>10.4f}')
print(f'  {"Answer Relevancy (RAGAS)":<32} {avg_r:>10.4f}')
print(f'  {"Context Precision (RAGAS)":<32} {avg_cp:>10.4f}')
print(f'  {"Context Recall (RAGAS)":<32} {avg_cr:>10.4f}')
print('=' * 60)
print('\nPer-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = df[df['ground_truth']==lbl]
    if len(sub):
        print(f'  {lbl:>5}: {sub["is_correct"].mean():.1%} ({int(sub["is_correct"].sum())}/{len(sub)}) '
              f'| faithfulness={sub["faithfulness"].mean():.3f}')
print('\nBaris untuk tabel skripsi:')
print(f'  | Baseline (BM25) | {label_acc:.3f} | {hallu_rate:.3f} | '
      f'{avg_f:.3f} | {avg_r:.3f} | {avg_cp:.3f} | {avg_cr:.3f} |')
print('\n' + '-' * 60)
print(f'PENGINGAT: Jalankan ulang dengan MAX_SAMPLES=1000 untuk evaluasi final!')
print('-' * 60)